## Homework 5 – Application of Hugging Face Models

## 1. Financial Phrasebank

In [7]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import numpy as np

# Cargar el conjunto de datos Financial Phrasebank (con permiso para ejecutar código remoto)
dataset = load_dataset("financial_phrasebank", "sentences_allagree", trust_remote_code=True)
frases = dataset['train']['sentence']
etiquetas_verdaderas = dataset['train']['label']  # 0: negativo, 1: neutral, 2: positivo

# Cargar el modelo FinBERT y el tokenizador
nombre_modelo = "ProsusAI/finbert"
tokenizador = AutoTokenizer.from_pretrained(nombre_modelo)
modelo = AutoModelForSequenceClassification.from_pretrained(nombre_modelo)

# Definir etiquetas de sentimiento
mapa_etiquetas = {0: "negativo", 1: "neutral", 2: "positivo"}



In [8]:
# Seleccionamos algunas frases (para no procesar todo de golpe)
frases_muestra = frases[:100]  # puedes ajustar el número

# Tokenizar las frases
tokens = tokenizador(frases_muestra, padding=True, truncation=True, return_tensors="pt")

# Desactivar el cálculo de gradientes para hacer inferencia (más eficiente)
with torch.no_grad():
    salidas = modelo(**tokens)

# Obtener las predicciones (logits → índice de clase con mayor puntuación)
predicciones = torch.argmax(salidas.logits, dim=1)

# Mostrar etiquetas predichas
etiquetas_predichas = predicciones.numpy()


In [9]:
# Mostrar frases con etiqueta verdadera y predicha
for i in range(10):
    print(f"Frase: {frases_muestra[i]}")
    print(f"Etiqueta verdadera: {mapa_etiquetas[etiquetas_verdaderas[i]]}")
    print(f"Etiqueta predicha : {mapa_etiquetas[etiquetas_predichas[i]]}")
    print("-" * 60)


Frase: According to Gran , the company has no plans to move all production to Russia , although that is where the company is growing .
Etiqueta verdadera: neutral
Etiqueta predicha : positivo
------------------------------------------------------------
Frase: For the last quarter of 2010 , Componenta 's net sales doubled to EUR131m from EUR76m for the same period a year earlier , while it moved to a zero pre-tax profit from a pre-tax loss of EUR7m .
Etiqueta verdadera: positivo
Etiqueta predicha : negativo
------------------------------------------------------------
Frase: In the third quarter of 2010 , net sales increased by 5.2 % to EUR 205.5 mn , and operating profit by 34.9 % to EUR 23.5 mn .
Etiqueta verdadera: positivo
Etiqueta predicha : negativo
------------------------------------------------------------
Frase: Operating profit rose to EUR 13.1 mn from EUR 8.7 mn in the corresponding period in 2007 representing 7.7 % of net sales .
Etiqueta verdadera: positivo
Etiqueta predich

In [10]:
from sklearn.metrics import accuracy_score

# Calcular precisión sobre las primeras 100 frases
precisión = accuracy_score(etiquetas_verdaderas[:100], etiquetas_predichas)
print(f"Precisión del modelo en 100 frases: {precisión:.2f}")


Precisión del modelo en 100 frases: 0.00


## 2. Practical Case

In [2]:
!pip install pymupdf transformers -q

In [1]:
pwd

'c:\\Users\\usuario\\Documents\\GitHub\\Spatial_Analysis_of_Peru-s_Protected_Areas\\241878_hw5'

In [5]:
import fitz  # PyMuPDF
from transformers import pipeline

# --- Paso 1: Cargar el PDF y extraer texto ---
ruta_pdf = r"C:\Users\usuario\Documents\GitHub\Spatial_Analysis_of_Peru-s_Protected_Areas\241878_hw5\Contrato_ejercicio.pdf"

doc = fitz.open(ruta_pdf)
texto_contrato = ""

for pagina in doc:
    texto_contrato += pagina.get_text()

# Verifica que haya texto
if not texto_contrato.strip():
    print("⚠️ No se extrajo texto del PDF.")
    exit()


In [6]:

# --- Paso 2: Usar modelo de resumen de Hugging Face ---
print("🔄 Generando resumen... esto puede tardar unos segundos.")

resumidor = pipeline("summarization", model="facebook/bart-large-cnn")

🔄 Generando resumen... esto puede tardar unos segundos.


In [7]:

# --- Paso 3: Dividir texto en partes de hasta 1000 caracteres aprox ---
def dividir_texto(texto, max_chars=1000):
    partes = []
    inicio = 0
    while inicio < len(texto):
        fin = inicio + max_chars
        if fin < len(texto):
            # Intentar dividir por el último punto para no cortar frases
            fin = texto.rfind('.', inicio, fin) + 1 or fin
        partes.append(texto[inicio:fin].strip())
        inicio = fin
    return partes

partes_texto = dividir_texto(texto_contrato)

resumen_final = ""

for i, parte in enumerate(partes_texto):
    try:
        resumen = resumidor(parte, max_length=130, min_length=40, do_sample=False)
        resumen_final += f"📝 Parte {i+1}: {resumen[0]['summary_text']}\n\n"
    except Exception as e:
        resumen_final += f"⚠️ Error al resumir parte {i+1}: {e}\n\n"

In [8]:
# --- Paso 4: Mostrar resultado ---
print("\n📄 Resumen completo del contrato:\n")
print(resumen_final)



📄 Resumen completo del contrato:

📝 Parte 1: Arrendamiento de LOCAL COMERCIAL. ARRENDAMIENTO de LOCal COMERCial. Arrendamiono o Alquiler del Local. Comercial que celebran, de una parte ZAMORA QUISPE, MARÍA ELENA, y de la otra parte, JULIO MENDOZA.

📝 Parte 2: El local comercial is ubicado en Av. Alameda del Norte N.º 451 y Av. Los Laureles Nº 448, en buen estado de conservación and habitabilidad. El piso de cerámica tipo porcelanato nuevo está obligado a entregar el local comerscial tal como se le está entregando.

📝 Parte 3: El monto de la renta que pagará EL. ARRENDATARIO O INQUILINO en calidad de contraprestación por. el uso  purposefullydel bien, asciende a la suma. de S/ 1400 (MIL CUATROCIENTOS)mensuales. La forma de pago de the renta será por mensualidades.

📝 Parte 4: Las partes convienen fijar un plazo de duración determinada para el presente contrato. El cual será de 12 meses que se computará a partir de la fecha de suscripción 11                 de Setiembre del 2024.

📝 Par